In [9]:
# === One-Cell: Hyperopt FT-Transformer with SNN head, CSV logging & final retrain ===
# - Optimizes recall @ FPR<=5% (VALID)
# - Logs per-epoch/per-trial to ftt_trials.csv and best-per-trial to ftt_trials_best.csv
# - Saves final best model + meta to ftt_export/

import os, sys, json, csv, math, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix, recall_score
import torch.nn.functional as F

# ensure optuna & tqdm
for pkg in ("optuna", "tqdm"):
    try:
        __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import optuna
from tqdm.auto import tqdm

# --- SNN imports (SpikingJelly) ---
try:
    from spikingjelly.activation_based import neuron, surrogate, functional
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spikingjelly"])
    from spikingjelly.activation_based import neuron, surrogate, functional

# ---------------- Config ----------------
CSV = Path("Variant V.csv")     # change path if needed
SEED = 42
WEIGHT_DECAY = 1e-5
FPR_CAP = 0.05

# Search settings (tune as you like)
N_TRIALS = 20          # increase to 60-100 for better results
TRIAL_EPOCHS = 8       # epochs per trial
FINAL_EPOCHS = 20      # retrain epochs for best config

# CSV logs
TRIALS_CSV = Path("ftt_trials_V5_snn.csv")
BEST_CSV   = Path("ftt_trials_best_V5_snn.csv")
FIELDS = ["trial","epoch","d_token","n_blocks","n_heads","ffn_hidden","dropout","lr","batch",
          "val_auc","val_prauc","val_recall_at_fpr","val_fpr","val_threshold"]

# Export dir
EXPORT_DIR = Path("ftt_export_V5_snn")

# ---------------- Repro/Device ----------------
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {device}")
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

# ---------------- Data prep ----------------
df = pd.read_csv(CSV)

# autodetect binary target (edit if needed)
for c in ["fraud_bool","fraud","is_fraud","label","target"]:
    if c in df.columns: target=c; break
else:
    raise ValueError("Set target column name.")

drop_like={"customer","id","uuid"}
feat=[c for c in df.columns if c!=target and not any(x in c.lower() for x in drop_like)]
cat=[c for c in feat if str(df[c].dtype) in ("object","category")]
for c in feat:
    if c not in cat and pd.api.types.is_integer_dtype(df[c]) and df[c].nunique()<=50:
        cat.append(c)
cont=[c for c in feat if c not in cat and pd.api.types.is_numeric_dtype(df[c])]

train_df, tmp = train_test_split(df, test_size=0.2, stratify=df[target], random_state=SEED)
valid_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp[target], random_state=SEED)

def prep(d):
    d=d.copy()
    for c in cat: d[c]=d[c].astype("category").cat.codes.astype("int64")
    Xc = torch.as_tensor(d[cat].values, dtype=torch.long)     if cat  else None
    Xn = torch.as_tensor(d[cont].values, dtype=torch.float32) if cont else None
    y  = torch.as_tensor(d[target].values, dtype=torch.float32).view(-1,1)
    return Xn,Xc,y

Xn_tr,Xc_tr,y_tr = prep(train_df)
Xn_va,Xc_va,y_va = prep(valid_df)
Xn_te,Xc_te,y_te = prep(test_df)

# normalize continuous by train stats
if Xn_tr is not None:
    m=Xn_tr.mean(0,keepdim=True); s=Xn_tr.std(0,keepdim=True).clamp_min(1e-6)
    Xn_tr=(Xn_tr-m)/s; Xn_va=(Xn_va-m)/s; Xn_te=(Xn_te-m)/s

def td(xn,xc,y):
    parts=[]
    if xn is not None: parts.append(xn.to(device))
    if xc is not None: parts.append(xc.to(device))
    parts.append(y.to(device)); return parts

# ---------------- Minimal FT-Transformer ----------------
class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features: int, cat_cardinalities, d_token: int):
        super().__init__()
        self.n_num = n_num_features
        self.cat_cardinalities = cat_cardinalities or []
        self.d_token = d_token
        if self.n_num > 0:
            self.num_weight = nn.Parameter(torch.randn(self.n_num, d_token) * 0.02)
            self.num_bias   = nn.Parameter(torch.zeros(self.n_num, d_token))
        else:
            self.register_parameter("num_weight", None)
            self.register_parameter("num_bias",   None)
        self.cat_embeds = nn.ModuleList([nn.Embedding(card, d_token) for card in self.cat_cardinalities])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)

    def forward(self, x_num, x_cat):
        if x_num is None and x_cat is None:
            raise ValueError("No features to tokenize")
        B = x_num.size(0) if x_num is not None else x_cat.size(0)
        toks=[]
        if x_num is not None:
            toks.append(x_num.unsqueeze(-1)*self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0))
        if x_cat is not None and len(self.cat_cardinalities)>0:
            emb = [e(x_cat[:,i]) for i,e in enumerate(self.cat_embeds)]
            toks.append(torch.stack(emb, dim=1))
        x = torch.cat(toks, dim=1)
        x = torch.cat([self.cls_token.expand(B,1,-1), x], dim=1)
        return x

# ---- SNN Head (Spiking Jelly) ----
class SNNHead(nn.Module):
    """
    CLS [B, d] -> 1 logit [B,1] using LIF dynamics with membrane readout.
    Uses T steps on the same input, averages membrane potential (smooth),
    then linear readout. Includes BN + learnable scale and bias current to
    avoid dead neurons at init.
    """
    def __init__(self, d_token, T=8, tau=2.0, v_th=0.2, v_reset=0.0):
        super().__init__()
        self.T = T
        self.bn = nn.BatchNorm1d(d_token, affine=True)
        self.fc_in = nn.Linear(d_token, d_token)
        # init a bit “hotter” so we’re near threshold from step 1
        nn.init.kaiming_normal_(self.fc_in.weight, nonlinearity='linear')
        nn.init.zeros_(self.fc_in.bias)

        self.lif = neuron.LIFNode(
            tau=tau,
            v_threshold=v_th,          # lower threshold to encourage activity
            v_reset=v_reset,
            surrogate_function=surrogate.ATan(),
            detach_reset=True
        )
        # learnable positive gain and small bias current
        self.log_alpha = nn.Parameter(torch.zeros(1))      # alpha = softplus(log_alpha)+1 ~ 1 at init
        self.bias_curr = nn.Parameter(torch.zeros(d_token))# small learnable bias current
        self.readout = nn.Linear(d_token, 1)
        nn.init.zeros_(self.readout.bias)

    def forward(self, cls_vec):  # [B, d]
        try:
            functional.reset_net(self.lif)
        except Exception:
            if hasattr(self.lif, "reset"):
                self.lif.reset()
        # BN helps keep scale healthy; training=True inside forward
        x = self.bn(cls_vec)
        x = self.fc_in(x)
        alpha = F.softplus(self.log_alpha) + 1.0
        x = x * alpha + self.bias_curr  # scale + bias current

        v_sum = 0.0
        for _ in range(self.T):
            _ = self.lif(x)            # updates internal membrane self.lif.v
            v_sum = v_sum + self.lif.v
        v_avg = v_sum / self.T         # [B, d]
        return self.readout(v_avg)     # [B, 1]

class FTTransformer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, d_token=32, n_blocks=4, n_heads=8, ffn_hidden=128, dropout=0.2, n_classes=1):
        super().__init__()
        self.tok = FeatureTokenizer(n_num_features, cat_cardinalities, d_token)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=ffn_hidden,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_blocks)

        # SNN prediction head → single logit
        self.head = SNNHead(d_token=d_token, T=8)

    def forward(self, x_num, x_cat):
        x = self.tok(x_num, x_cat)    # [B, 1+F, d]
        x = self.encoder(x)
        cls = x[:,0,:]                # [B, d]
        return self.head(cls)         # [B, 1]

n_num = 0 if Xn_tr is None else Xn_tr.shape[1]
cards = [int(df[c].astype("category").cat.categories.size) for c in cat] if cat else None

# ---------------- Metrics helper ----------------
def recall_at_fpr_cap(y_true_tensor, y_prob, cap=FPR_CAP):
    y_true = y_true_tensor.cpu().numpy().ravel()
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    ok = fpr <= cap
    if not np.any(ok):  # no threshold meets cap
        return 0.0, 0.5, 1.0
    i = np.argmax(tpr[ok])
    return tpr[ok][i], thr[ok][i], fpr[ok][i]

# ---------------- Objective ----------------
def objective(trial: optuna.Trial):
    # search space
    d_token   = trial.suggest_categorical("d_token", [16, 32, 48, 64])
    n_blocks  = trial.suggest_int("n_blocks", 2, 6)
    n_heads   = trial.suggest_categorical("n_heads", [4, 8])
    ffn_hid   = trial.suggest_categorical("ffn_hidden", [64, 128, 192, 256])
    dropout   = trial.suggest_float("dropout", 0.0, 0.4)
    lr        = trial.suggest_float("lr", 3e-4, 3e-3, log=True)
    batch_sz  = trial.suggest_categorical("batch", [1024, 2048, 4096, 8192])

    net = FTTransformer(n_num, cards, d_token, n_blocks, n_heads, ffn_hid, dropout, n_classes=1).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    train_loader = DataLoader(TensorDataset(*td(Xn_tr,Xc_tr,y_tr)), batch_size=batch_sz, shuffle=True)
    valid_loader = DataLoader(TensorDataset(*td(Xn_va,Xc_va,y_va)), batch_size=batch_sz*2)

    def predict(loader):
        net.eval(); out=[]
        with torch.no_grad():
            for *xs,_ in loader:
                out.append(torch.sigmoid(net(*xs)).squeeze(1).cpu())
        return torch.cat(out).numpy()

    best_epoch_rec = -1.0
    best_row = None

    for ep in range(1, TRIAL_EPOCHS+1):
        # train one epoch
        net.train()
        for *xs, y in train_loader:
            logits = net(*xs)                  # [B,1]
            loss = loss_fn(logits, y)          # y: [B,1]
            opt.zero_grad(); loss.backward(); opt.step()

        # validate and log
        p_va = predict(valid_loader)
        auc   = roc_auc_score(y_va.cpu().numpy(), p_va)
        prauc = average_precision_score(y_va.cpu().numpy(), p_va)
        rec, thr, fpr = recall_at_fpr_cap(y_va, p_va, FPR_CAP)

        row = {
            "trial": trial.number, "epoch": ep,
            "d_token": d_token, "n_blocks": n_blocks, "n_heads": n_heads,
            "ffn_hidden": ffn_hid, "dropout": dropout, "lr": lr, "batch": batch_sz,
            "val_auc": auc, "val_prauc": prauc, "val_recall_at_fpr": rec, "val_fpr": fpr, "val_threshold": thr
        }
        # append per-epoch row
        write_header = not TRIALS_CSV.exists()
        with open(TRIALS_CSV, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=FIELDS)
            if write_header: w.writeheader()
            w.writerow(row)

        # keep best-epoch row for this trial
        if rec > best_epoch_rec:
            best_epoch_rec = rec
            best_row = row

        trial.report(best_epoch_rec, ep)
        if trial.should_prune():
            break

    # write best-epoch-per-trial row
    write_header = not BEST_CSV.exists()
    with open(BEST_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if write_header: w.writeheader()
        w.writerow(best_row)

    return best_epoch_rec

# ---------------- Run study ----------------
study = optuna.create_study(direction="maximize", study_name="ftt_recall_at_fpr_cap")
print("Running hyperparameter search...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best recall@FPR<=5%:", study.best_value)
print("Best params:", study.best_params)

# ---------------- Retrain best config ----------------
bp = study.best_params
net = FTTransformer(
    n_num, cards,
    d_token=bp["d_token"], n_blocks=bp["n_blocks"], n_heads=bp["n_heads"],
    ffn_hidden=bp["ffn_hidden"], dropout=bp["dropout"], n_classes=1
).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=bp["lr"], weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCEWithLogitsLoss()

train_loader = DataLoader(TensorDataset(*td(Xn_tr,Xc_tr,y_tr)), batch_size=bp["batch"], shuffle=True)
valid_loader = DataLoader(TensorDataset(*td(Xn_va,Xc_va,y_va)), batch_size=bp["batch"]*2)
test_loader  = DataLoader(TensorDataset(*td(Xn_te,Xc_te,y_te)), batch_size=bp["batch"]*2)

def predict(loader):
    net.eval(); out=[]
    with torch.no_grad():
        for *xs,_ in loader:
            out.append(torch.sigmoid(net(*xs)).squeeze(1).cpu())
    return torch.cat(out).numpy()

best_rec, best_state = -1.0, None
for ep in range(1, FINAL_EPOCHS+1):
    net.train()
    for *xs, y in train_loader:
        logits = net(*xs)            # [B,1]
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()

    p_va = predict(valid_loader)
    rec, thr_star, _ = recall_at_fpr_cap(y_va, p_va, FPR_CAP)
    if rec > best_rec:
        best_rec, best_state = rec, {k:v.detach().cpu().clone() for k,v in net.state_dict().items()}
    if ep % 2 == 0:
        print(f"[Retrain] Epoch {ep:02d} | VALID recall@FPR<=5% = {rec:.4f}")

net.load_state_dict(best_state)

# ---------------- Final evaluation ----------------
p_va = predict(valid_loader)
p_te = predict(test_loader)
print("\nVALID AUC:", roc_auc_score(y_va.cpu().numpy(), p_va))
print("VALID PR-AUC:", average_precision_score(y_va.cpu().numpy(), p_va))
rec_va, thr_star, fpr_va = recall_at_fpr_cap(y_va, p_va, FPR_CAP)
print(f"VALID recall@FPR<={int(FPR_CAP*100)}%: {rec_va:.4f} | chosen_thr: {thr_star:.6f} | FPR: {fpr_va:.4f}")

# apply same threshold to test
y_te_np = y_te.cpu().numpy().ravel()
yhat_te = (p_te >= thr_star).astype(int)
tn,fp,fn,tp = confusion_matrix(y_te_np, yhat_te).ravel()
rec_te = tp/(tp+fn+1e-12); fpr_te = fp/(fp+tn+1e-12)
print(f"TEST  recall: {rec_te:.4f} | TEST FPR: {fpr_te:.4f}")
print("TEST  AUC:", roc_auc_score(y_te.cpu().numpy(), p_te))
print("TEST  PR-AUC:", average_precision_score(y_te.cpu().numpy(), p_te))

# ---------------- Save artifacts ----------------
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(net.state_dict(), EXPORT_DIR/"model.pt")
with open(EXPORT_DIR/"meta.json","w") as f:
    json.dump({
        "target": target,
        "cat_cols": cat,
        "cont_cols": cont,
        "fpr_cap": FPR_CAP,
        "threshold": float(thr_star),
        **study.best_params
    }, f, indent=2)

print(f"\nSaved model + meta to {EXPORT_DIR}/")
print(f"Per-epoch trial log: {TRIALS_CSV}")
print(f"Best-per-trial log : {BEST_CSV}")

Torch 2.5.1+cu121 | CUDA: True | Device: cuda
GPU: NVIDIA A100-SXM4-40GB


[I 2025-08-20 17:29:59,875] A new study created in memory with name: ftt_recall_at_fpr_cap


Running hyperparameter search...


  0%|          | 0/20 [00:00<?, ?it/s]

/home/naveed-online/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


...

[I 2025-08-20 17:57:53,050] Trial 18 finished with value: 0.37262012692656393 and parameters: {'d_token': 48, 'n_blocks': 2, 'n_heads': 4, 'ffn_hidden': 64, 'dropout': 0.38542613238265666, 'lr': 0.0004927011023605552, 'batch': 4096}. Best is trial 13 with value: 0.6600181323662738.


/home/naveed-online/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[I 2025-08-20 17:58:31,236] Trial 19 finished with value: 0.6328195829555757 and parameters: {'d_token': 48, 'n_blocks': 4, 'n_heads': 8, 'ffn_hidden': 256, 'dropout': 0.11806255004729081, 'lr': 0.0020834552691967874, 'batch': 4096}. Best is trial 13 with value: 0.6600181323662738.
Best recall@FPR<=5%: 0.6600181323662738
Best params: {'d_token': 48, 'n_blocks': 3, 'n_heads': 8, 'ffn_hidden': 64, 'dropout': 0.0007366790137155743, 'lr': 0.000636502055105332, 'batch': 4096}


/home/naveed-online/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[Retrain] Epoch 02 | VALID recall@FPR<=5% = 0.6174
[Retrain] Epoch 04 | VALID recall@FPR<=5% = 0.6401
[Retrain] Epoch 06 | VALID recall@FPR<=5% = 0.6346
[Retrain] Epoch 08 | VALID recall@FPR<=5% = 0.6482
[Retrain] Epoch 10 | VALID recall@FPR<=5% = 0.6528
[Retrain] Epoch 12 | VALID recall@FPR<=5% = 0.6464
[Retrain] Epoch 14 | VALID recall@FPR<=5% = 0.6510
[Retrain] Epoch 16 | VALID recall@FPR<=5% = 0.6609
[Retrain] Epoch 18 | VALID recall@FPR<=5% = 0.6464
[Retrain] Epoch 20 | VALID recall@FPR<=5% = 0.6383

VALID AUC: 0.9188568725370849
VALID PR-AUC: 0.41193686306570276
VALID recall@FPR<=5%: 0.6609 | chosen_thr: 0.040992 | FPR: 0.0496
TEST  recall: 0.6410 | TEST FPR: 0.0470
TEST  AUC: 0.9228116496671799
TEST  PR-AUC: 0.426505455282501

Saved model + meta to ftt_export_V5_snn/
Per-epoch trial log: ftt_trials_V5_snn.csv
Best-per-trial log : ftt_trials_best_V5_snn.csv
